# AQE
AQE (Adaptive Query Execution) is a feature introduced in Spark 3.0 that allows Spark to optimize query plans at runtime, based on actual execution statistics (like row counts, data sizes), rather than just static estimates from the Catalyst Optimizer.

### Features-of-AQE
1. Dynamically coalescing shuffle partition
2. Dynamically switching join strategy
3. Dynamically optimizing skew join
---

1. **Dynamically Coalescing Shuffle Partitions**
   * Reduces the number of shuffle partitions based on actual data sizes after shuffle.
   * Helps avoid small, inefficient tasks and improves resource utilization.

2. **Dynamically Switching Join Strategy**
   * Chooses the most efficient join type at runtime (e.g., broadcast join, sort-merge join).
   * Based on actual table sizes observed during query execution.

3. **Dynamically Optimizing Skew Joins**
   * Detects data skew at runtime.
   * Splits skewed partitions to balance the workload across tasks and prevent slow or straggling tasks.

## Enable AQE in PySpark

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AQE-DEMO") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.join.enabled", "true") \
    .getOrCreate()

In [9]:
spark

## 1. Dynamically Coalescing Shuffle Partitions

In [10]:
# Generate data with intentional size variation
data = [(i % 1000, i * 10) for i in range(1000000)]  # 1M rows
df = spark.createDataFrame(data, ["id", "value"]).repartition(100) 
print(f"Count of partition: {df.rdd.getNumPartitions()}")

# Group by - triggers shuffle and coalescing
result = df.groupBy("id").sum("value")

# Check final partition count (will be reduced by AQE)
print(f"Final partition count: {result.rdd.getNumPartitions()}")
result.explain()  # Look for "AQEShuffleRead coalesced" in physical plan

Count of partition: 100
Final partition count: 2
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- *(3) HashAggregate(keys=[id#32L], functions=[sum(value#33L)])
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 1
            +- Exchange hashpartitioning(id#32L, 200), ENSURE_REQUIREMENTS, [plan_id=297]
               +- *(2) HashAggregate(keys=[id#32L], functions=[partial_sum(value#33L)])
                  +- ShuffleQueryStage 0
                     +- Exchange RoundRobinPartitioning(100), REPARTITION_BY_NUM, [plan_id=275]
                        +- *(1) Scan ExistingRDD[id#32L,value#33L]
+- == Initial Plan ==
   HashAggregate(keys=[id#32L], functions=[sum(value#33L)])
   +- Exchange hashpartitioning(id#32L, 200), ENSURE_REQUIREMENTS, [plan_id=265]
      +- HashAggregate(keys=[id#32L], functions=[partial_sum(value#33L)])
         +- Exchange RoundRobinPartitioning(100), REPARTITION_BY_NUM, [plan_id=261]
            +- Sc

### Explanation:
- Spark will initially create 100 shuffle partitions(instructed by user).
- AQE may coalesce them down (e.g., to 2) based on actual data written per partition.

## 2. Dynamically Switching Join Strategy

In [6]:
# Large table
big_df = spark.range(0, 1_000_000).withColumnRenamed("id", "big_id")

# Small table (will be broadcasted dynamically)
small_df = spark.range(0, 10).withColumnRenamed("id", "small_id")

# Join
joined_df = big_df.join(small_df, big_df["big_id"] == small_df["small_id"])

joined_df.explain(True)

== Parsed Logical Plan ==
Join Inner, (big_id#25L = small_id#27L)
:- Project [id#24L AS big_id#25L]
:  +- Range (0, 1000000, step=1, splits=Some(4))
+- Project [id#26L AS small_id#27L]
   +- Range (0, 10, step=1, splits=Some(4))

== Analyzed Logical Plan ==
big_id: bigint, small_id: bigint
Join Inner, (big_id#25L = small_id#27L)
:- Project [id#24L AS big_id#25L]
:  +- Range (0, 1000000, step=1, splits=Some(4))
+- Project [id#26L AS small_id#27L]
   +- Range (0, 10, step=1, splits=Some(4))

== Optimized Logical Plan ==
Join Inner, (big_id#25L = small_id#27L)
:- Project [id#24L AS big_id#25L]
:  +- Range (0, 1000000, step=1, splits=Some(4))
+- Project [id#26L AS small_id#27L]
   +- Range (0, 10, step=1, splits=Some(4))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [big_id#25L], [small_id#27L], Inner, BuildRight, false
   :- Project [id#24L AS big_id#25L]
   :  +- Range (0, 1000000, step=1, splits=4)
   +- BroadcastExchange HashedRelationBroadcastMode(List(

### Explanation:
- Even if Spark couldn’t know in advance that small_df is small enough and it will apply sort merge join by default.
- AQE uses runtime statistics to switch to Broadcast Hash Join for better performance.

## 3. Dynamically Optimizing Skew Joins

In [15]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)
# Create skewed data (key "0" has 50% of data)
skewed_data = [("0", f"data_{i}") for i in range(500000)] + \
              [(str(i), f"data_{i}") for i in range(1, 500000)]
normal_data = [(str(i % 1000), f"info_{i}") for i in range(1000000)]

df1 = spark.createDataFrame(skewed_data, ["key", "skewed_data"])
df2 = spark.createDataFrame(normal_data, ["key", "normal_data"])

# Join with skew optimization
joined = df1.join(df2, "key").explain()
# joined.count()  # Action triggers execution
# joined.explain()

# Check for skew handling in Spark UI (look for additional tasks on skewed keys)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#80, skewed_data#81, normal_data#83]
   +- SortMergeJoin [key#80], [key#82], Inner
      :- Sort [key#80 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(key#80, 200), ENSURE_REQUIREMENTS, [plan_id=1025]
      :     +- Filter isnotnull(key#80)
      :        +- Scan ExistingRDD[key#80,skewed_data#81]
      +- Sort [key#82 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(key#82, 200), ENSURE_REQUIREMENTS, [plan_id=1026]
            +- Filter isnotnull(key#82)
               +- Scan ExistingRDD[key#82,normal_data#83]




### Explanation:
- `AdaptiveSparkPlan` is enable
- `isFinalPlan=false` means Spark hasn't executed yet.
- AQE collects shuffle statistics during the first stages of execution.
- The physical plan will be re-optimized after these stages complete.
- Key takeaway: This is a pre-execution plan. AQE will rewrite it during execution based on real data statistics.